# Interactive EDA (Plotly) — Predicting Smartphone Addiction

Companion to `eda.ipynb`. That notebook covers the full analysis with static
matplotlib/seaborn plots; this one re-renders the key visuals as **interactive Plotly
figures** (hover tooltips, feature-picker dropdowns, zoom/pan) and adds a couple of views
that only make sense interactively (parallel coordinates). It's kept as a separate file so
the static notebook stays readable and doesn't get weighed down by the larger, JS-heavy
Plotly outputs.

Same dataset, same conclusions — see `eda.ipynb` for the narrative and written
observations.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 50)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
TARGET = "addicted_label"

# A fixed sample for the plots that need row-level data (box/violin, parallel coords).
# Plotting all ~691K rows directly would bloat the notebook file and the browser DOM for
# little visual benefit at this density, so we sample for those views; the histograms and
# bar charts below use full-data aggregates instead.
sample = train.sample(n=20_000, random_state=42)

print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
print(f"row-level sample size: {len(sample):,}")

train shape: (691369, 14)
test shape:  (296302, 13)
row-level sample size: 20,000


## 1. Missing values — train vs. test

In [2]:
missing = pd.DataFrame({
    "train": train.isnull().mean().mul(100),
    "test": test.reindex(columns=train.columns).isnull().mean().mul(100),
}).drop(index=["id", TARGET], errors="ignore")
missing = missing.sort_values("train").round(2)

fig = go.Figure()
fig.add_bar(y=missing.index, x=missing["train"], name="train", orientation="h",
            marker_color="#440154", hovertemplate="%{y}: %{x:.2f}%<extra>train</extra>")
fig.add_bar(y=missing.index, x=missing["test"], name="test", orientation="h",
            marker_color="#21918c", hovertemplate="%{y}: %{x:.2f}%<extra>test</extra>")
fig.update_layout(
    title="Missing values per feature",
    xaxis_title="% missing", barmode="group",
    height=450, legend_title="split",
)
fig.show()

## 2. Target distribution

In [3]:
counts = train[TARGET].value_counts().sort_index()
labels = ["Not addicted (0)", "Addicted (1)"]

fig = go.Figure(go.Pie(
    labels=labels, values=counts.values, hole=0.55,
    marker_colors=["#440154", "#21918c"],
    textinfo="label+percent", hovertemplate="%{label}: %{value:,}<extra></extra>",
))
fig.update_layout(title="Target distribution (addicted_label)", height=450,
                   annotations=[dict(text=f"n={len(train):,}", x=0.5, y=0.5,
                                      font_size=14, showarrow=False)])
fig.show()

## 3. Numeric feature distributions

Full-data histograms (not the sample), split by target class. Use the dropdown to switch
between features.

In [4]:
fig = go.Figure()
n_bins = 40
buttons = []

for i, col in enumerate(NUM_COLS):
    vals = train[[col, TARGET]].dropna()
    lo, hi = vals[col].min(), vals[col].max()
    bins = np.linspace(lo, hi, n_bins + 1)
    centers = (bins[:-1] + bins[1:]) / 2
    width = bins[1] - bins[0]

    for cls, color, name in [(0, "#440154", "Not addicted"), (1, "#21918c", "Addicted")]:
        sub = vals.loc[vals[TARGET] == cls, col]
        counts, _ = np.histogram(sub, bins=bins, density=True)
        fig.add_bar(x=centers, y=counts, width=width, name=name, marker_color=color,
                    opacity=0.65, visible=(i == 0),
                    hovertemplate=f"{col}=%{{x:.2f}}<br>density=%{{y:.3f}}<extra>{name}</extra>")

    visible = [False] * (2 * len(NUM_COLS))
    visible[2 * i] = True
    visible[2 * i + 1] = True
    buttons.append(dict(label=col, method="update",
                         args=[{"visible": visible}, {"title": f"Distribution of {col} by target"}]))

fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons, x=1.0, xanchor="right", y=1.18, yanchor="top")],
    barmode="overlay",
    title=f"Distribution of {NUM_COLS[0]} by target",
    xaxis_title="value", yaxis_title="density", height=500, legend_title="class",
)
fig.show()

## 4. Categorical feature addiction rate

In [5]:
fig = make_subplots(rows=1, cols=len(CAT_COLS), subplot_titles=CAT_COLS)
overall_rate = train[TARGET].mean()

for i, col in enumerate(CAT_COLS, start=1):
    rate = train.groupby(col, observed=True)[TARGET].mean().sort_values()
    fig.add_bar(x=rate.values, y=rate.index, orientation="h",
                marker_color="#31688e", showlegend=False, row=1, col=i,
                hovertemplate="%{y}: %{x:.1%}<extra></extra>")
    fig.add_vline(x=overall_rate, line_dash="dash", line_color="grey", row=1, col=i)

fig.update_xaxes(title_text="addiction rate", tickformat=".0%")
fig.update_layout(title="Addiction rate by category (dashed line = overall rate)", height=420)
fig.show()

## 5. Correlation matrix

In [6]:
corr = train[NUM_COLS + [TARGET]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale="Viridis", zmid=0, text=corr.values, texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}: %{z}<extra></extra>",
))
fig.update_layout(title="Correlation matrix (numeric features + target)", height=550,
                   yaxis_autorange="reversed")
fig.show()

## 6. Boxplots for the strongest signals

Computed on the 20K-row sample (box quartiles are stable at this size, and it keeps the
notebook light).

In [7]:
top_features = ["notifications_per_day", "app_opens_per_day", "weekend_screen_time", "sleep_hours"]
fig = make_subplots(rows=1, cols=len(top_features), subplot_titles=top_features)

for i, col in enumerate(top_features, start=1):
    for cls, color, name in [(0, "#440154", "Not addicted"), (1, "#21918c", "Addicted")]:
        fig.add_box(y=sample.loc[sample[TARGET] == cls, col], name=name, marker_color=color,
                    showlegend=(i == 1), boxpoints=False, row=1, col=i)

fig.update_layout(title="Feature vs. target (sampled, n=20,000)", height=450)
fig.show()

## 7. Parallel coordinates — top features colored by addiction

An interactive-only view: drag along any axis to brush a range and see how it filters the
other axes. Lines are colored by `addicted_label`.

In [8]:
pc_cols = ["notifications_per_day", "app_opens_per_day", "weekend_screen_time",
           "sleep_hours", "social_media_hours"]
pc_data = sample[pc_cols + [TARGET]].dropna()

fig = px.parallel_coordinates(
    pc_data, dimensions=pc_cols, color=TARGET,
    color_continuous_scale="Viridis",
    title="Parallel coordinates — top features (sampled, n=%d)" % len(pc_data),
)
fig.update_layout(height=500)
fig.show()

## Notes

- All figures here reproduce findings already written up in `eda.ipynb` — the value add is
  interactivity (hover for exact values, the feature-picker dropdown in §3, brushing in the
  parallel coordinates plot), not new conclusions.
- Histograms and the missing-value/categorical bar charts are built from **full-data
  aggregates** (`np.histogram`, `groupby`), so they're exact. The box plots and parallel
  coordinates use a fixed 20,000-row random sample (`random_state=42`) since Plotly embeds
  row-level data directly in the notebook JSON — plotting all ~691K rows there would make
  the file large and sluggish to open for no real visual gain.
- See `eda.ipynb` §9 for the written summary of takeaways and how they fed into feature
  engineering.